# Day 16: PySpark Batch Pipelines and SQL
**An end-to-end pipeline, a SQL refresher, Spark SQL and consolidation of Weeks 1 to 3**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.

## 1. Setup

In [ ]:
# ===== Option A: Google Colab (active) =====
!pip install -q pyspark

# ===== Option B: VS Code / local =====
# Java 17 + pip install pyspark pandas   (see Day 14)
import sqlite3, time
import pandas as pd
from pyspark.sql import SparkSession, functions as F
spark = (SparkSession.builder.master("local[*]").appName("day16")
         .config("spark.sql.shuffle.partitions", "4").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

## 2. Shared sample data

In [ ]:
orders_rows = [
    (1001, 1, 250.50, "2026-11-12", "Bengaluru"), (1002, 2, 1200.0, "2026-11-12", "Tokyo"),
    (1003, 3, 410.0, "2026-11-12", "Osaka"), (1005, 5, 560.0, "2026-11-13", "Chennai"),
    (1006, 6, 90.25, "2026-11-13", "Kyoto"), (1007, 1, 3100.0, "2026-11-13", "Bengaluru"),
    (1008, 2, 3300.0, "2026-11-12", "Tokyo"), (1009, 99, 780.0, "2026-11-14", "Pune"),
    (1010, 98, 150.0, "2026-11-14", "Osaka"), (1011, 3, 2000.0, "2026-11-14", "Osaka"),
]
customer_rows = [(1, "Asha", "retail"), (2, "Kenji", "enterprise"), (3, "Mei", "retail"),
                 (5, "Priya", "enterprise"), (6, "Hana", "retail")]
ocols = ["order_id", "customer_id", "amount", "order_date", "city"]
ccols = ["customer_id", "name", "segment"]

conn = sqlite3.connect(":memory:")
pd.DataFrame(orders_rows, columns=ocols).to_sql("orders", conn, index=False)
pd.DataFrame(customer_rows, columns=ccols).to_sql("customers", conn, index=False)
q = lambda sql: pd.read_sql(sql, conn)
print("SQLite ready")

## 3. SQL refresher in SQLite

In [ ]:
# Business question: which cities bring in more than 1000 in revenue?
q('''
SELECT city, COUNT(*) AS orders, ROUND(SUM(amount), 2) AS revenue
FROM orders
GROUP BY city
HAVING SUM(amount) > 1000
ORDER BY revenue DESC
''')

In [ ]:
# Business question: revenue per customer segment, including unknown customers
q('''
SELECT COALESCE(c.segment, 'unknown') AS segment, ROUND(SUM(o.amount), 2) AS revenue
FROM orders o
LEFT JOIN customers c ON o.customer_id = c.customer_id
GROUP BY COALESCE(c.segment, 'unknown')
ORDER BY revenue DESC
''')

In [ ]:
# Business question: the top city on each day (CTE + window function)
top_city_sql = '''
WITH daily AS (
    SELECT order_date, city, SUM(amount) AS revenue
    FROM orders
    GROUP BY order_date, city
),
ranked AS (
    SELECT *, RANK() OVER (PARTITION BY order_date ORDER BY revenue DESC) AS rnk
    FROM daily
)
SELECT order_date, city, revenue FROM ranked WHERE rnk = 1 ORDER BY order_date
'''
sqlite_result = q(top_city_sql)
sqlite_result

### TODO 1: Write a query with `LAG` that shows each day's total revenue and the change from the previous day.

In [ ]:
# TODO: write your code here

## 4. The same query in Spark SQL

In [ ]:
orders = spark.createDataFrame(orders_rows, ocols)
customers = spark.createDataFrame(customer_rows, ccols)
orders.createOrReplaceTempView("orders")
customers.createOrReplaceTempView("customers")

spark_result = spark.sql(top_city_sql)
spark_result.show()
same = [tuple(r) for r in spark_result.collect()] == [tuple(r) for r in sqlite_result.itertuples(index=False)]
print("Spark result == SQLite result:", same)

## 5. A production-style pipeline

In [ ]:
%%writefile pipeline.py
import time
from pyspark.sql import DataFrame, functions as F

class PipelineError(Exception):
    pass

def to_silver(bronze: DataFrame) -> DataFrame:
    return (bronze.dropDuplicates(["order_id"])
            .filter(F.col("amount").isNotNull() & (F.col("amount") > 0))
            .withColumn("city", F.trim("city")))

def check_not_empty(df: DataFrame, name: str):
    if df.count() == 0:
        raise PipelineError(f"{name} is empty")

def check_unique(df: DataFrame, key: str):
    if df.count() != df.select(key).distinct().count():
        raise PipelineError(f"duplicate {key}")

def to_gold(spark, silver: DataFrame, customers: DataFrame) -> DataFrame:
    silver.createOrReplaceTempView("silver")
    customers.createOrReplaceTempView("dim_customers")
    return spark.sql('''
        SELECT s.order_date, s.city, COALESCE(c.segment, 'unknown') AS segment,
               COUNT(*) AS orders, ROUND(SUM(s.amount), 2) AS revenue
        FROM silver s LEFT JOIN dim_customers c ON s.customer_id = c.customer_id
        GROUP BY s.order_date, s.city, COALESCE(c.segment, 'unknown')
    ''')

def check_reconciles(silver: DataFrame, gold: DataFrame):
    s = silver.agg(F.round(F.sum("amount"), 2)).first()[0]
    g = gold.agg(F.round(F.sum("revenue"), 2)).first()[0]
    if abs(s - g) > 0.01:
        raise PipelineError(f"gold {g} != silver {s}")

def run(spark, orders: DataFrame, customers: DataFrame, run_date: str, out_path: str, log: list):
    start = time.perf_counter()
    bronze = orders.filter(F.col("order_date") == run_date)
    silver = to_silver(bronze)
    check_not_empty(silver, "silver")
    check_unique(silver, "order_id")
    gold = to_gold(spark, silver, customers)
    check_reconciles(silver, gold)
    gold.write.mode("overwrite").parquet(f"{out_path}/order_date={run_date}")
    entry = {"run_date": run_date, "bronze": bronze.count(), "silver": silver.count(),
             "gold": gold.count(), "seconds": round(time.perf_counter() - start, 2)}
    log.append(entry)
    return entry

In [ ]:
import importlib, pipeline
importlib.reload(pipeline)
run_log = []
for d in ["2026-11-12", "2026-11-13", "2026-11-14"]:
    print(pipeline.run(spark, orders, customers, d, "lake/gold/daily", run_log))
print("Re-run:", pipeline.run(spark, orders, customers, "2026-11-13", "lake/gold/daily", run_log))
pd.DataFrame(run_log)

In [ ]:
gold_all = spark.read.parquet("lake/gold/daily")
gold_all.orderBy("order_date", F.desc("revenue")).show()
print("Gold rows:", gold_all.count())

## 6. Tests for the quality gates

In [ ]:
%%writefile test_pipeline.py
import pytest
from pyspark.sql import SparkSession
import pipeline

@pytest.fixture(scope="session")
def spark():
    s = SparkSession.builder.master("local[1]").appName("tests").getOrCreate()
    yield s

def test_silver_removes_bad_and_duplicate_rows(spark):
    df = spark.createDataFrame([(1, 1, 10.0, "d", " A "), (1, 1, 10.0, "d", " A "), (2, 1, -5.0, "d", "B")],
                               ["order_id", "customer_id", "amount", "order_date", "city"])
    out = pipeline.to_silver(df).collect()
    assert len(out) == 1 and out[0].city == "A"

def test_empty_raises(spark):
    df = spark.createDataFrame([], "order_id INT")
    with pytest.raises(pipeline.PipelineError):
        pipeline.check_not_empty(df, "silver")

def test_duplicates_raise(spark):
    df = spark.createDataFrame([(1,), (1,)], ["order_id"])
    with pytest.raises(pipeline.PipelineError):
        pipeline.check_unique(df, "order_id")

In [ ]:
# ===== Option A: Google Colab (active) =====
!python -m pytest -q test_pipeline.py

# ===== Option B: VS Code =====
# Terminal: pytest -q test_pipeline.py

### TODO 2: Add a test for `check_reconciles` that builds a gold DataFrame whose revenue does not match silver and expects a `PipelineError`.

In [ ]:
# TODO: write your code here

---
**Ganbatte** (頑張って): *Do your best*

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)